# Where should a numerical electrostatic domain end?

A finite mesh needs an outer boundary even when the physical field extends to infinity. Solve the artificial-boundary error exactly for a dipole and test whether enlarging the numerical domain or matching the exterior field is more effective.

## Match a harmonic in an empty outer shell

At the surface of a source-containing sphere of radius $a$ prescribe $V(a,\theta)=V_a\cos\theta$. Outside this sphere there is no charge. Insert $V=f(r)\cos\theta$ in Laplace's equation. The angular Laplacian supplies the eigenvalue $-2$, so $(r^2 f')'-2f=0$. A power-law trial gives $f=Ar+B/r^2$. Decay at infinity removes the growing term and the open-space reference is $f_\infty=V_a(a/r)^2$.

Terminate the calculation at $R$ and ground the artificial wall. The two boundary equations instead give

$$f_R(r)=V_a\left(\frac ar\right)^2\frac{1-(r/R)^3}{1-(a/R)^3}.$$

A well-converged interior solve can reproduce this answer perfectly while retaining a systematic exterior-domain bias. An alternative boundary condition is $f'(R)+2f(R)/R=0$. It admits the decaying dipole and annihilates the unwanted growing coefficient. A single local coefficient matches one spherical-harmonic order exactly.

## Estimate the error before refining the mesh

At $r=2a$ with $R=5a$, the grounded boundary gives $f_R/f_\infty=(1-8/125)/(1-1/125)=117/124$. The amplitude error is about minus 5.65 percent. At a fixed observation radius, the leading bias decreases with the inverse cube of $R$. Refining the interior grid at fixed $R$ cannot remove it. Compare a second-order radial difference solution with both analytic curves to measure the two errors independently.
To see what the linear solver actually controls, put a uniform grid on $r/a$. Approximate the outgoing flux $r^2 f'$ at the right face by $r_{i+1/2}^2(f_{i+1}-f_i)/h$ and form the same quantity at the left face. Subtract the fluxes, divide by $h$ and equate to $2f_i$. The resulting tridiagonal system has positive diagonal $2+(r_{i+1/2}^2+r_{i-1/2}^2)/h^2$. Move the known inner and outer boundary terms to the right-hand side. For the grounded wall the outer contribution is zero; the inner value is the amplitude unit.

At $r/a=2$ the open-domain amplitude is 0.2500. The grounded reference gives 0.2359 when $R/a=5$. Repeating at $R/a=10$ raises that prediction to 0.24825. Keep the physical grid spacing fixed when making this domain comparison. If the cell count stays fixed, increasing the radius also coarsens the mesh, and the two effects can mask each other. The first figure supplies both analytical references needed to separate them.

## Set the experimental inputs

Edit these values, then run the calculation and both figures again.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

teal, orange, blue = "#19596a", "#aa462a", "#3868a0"
def axes(xlabel, ylabel):
    fig, ax = plt.subplots(figsize=(7, 4), constrained_layout=True)
    ax.set(xlabel=xlabel, ylabel=ylabel)
    ax.grid(alpha=0.2)
    return fig, ax

In [ ]:
outer_ratio = 5.0            # R / a
probe_ratio = 2.0            # observation radius / a
cells = 80

In [ ]:
from scipy.linalg import solve_banded
assert 1<probe_ratio<outer_ratio and cells>=8
r=np.linspace(1,outer_ratio,cells+1)
h=r[1]-r[0]
mid=(r[:-1]+r[1:])/2
lower,upper=mid[:-1]**2/h**2,mid[1:]**2/h**2
diagonal=lower+upper+2
bands=np.zeros((3,cells-1))
bands[1]=diagonal
bands[0,1:]=-upper[:-1]
bands[2,:-1]=-lower[1:]
rhs=np.zeros(cells-1)
rhs[0]=lower[0]
numeric=np.r_[1.,solve_banded((1,1),bands,rhs),0.]
def ratio(x,R):
    return (1-(x/R)**3)/(1-R**-3)
finite=ratio(r,outer_ratio)/r**2
reference=r**-2
print(f"Exact truncation bias at probe: {100*(ratio(probe_ratio,outer_ratio)-1):.3f}%")
print(f"Maximum field-amplitude discretisation error: {max(abs(numeric-finite)):.4g}")

In [ ]:
fig,ax=axes("Radius r/a","Dipole amplitude / prescribed value")
ax.plot(r,reference,color=orange,label="Open domain")
ax.plot(r,finite,color=teal,label="Grounded exterior")
ax.plot(r[::max(1,cells//15)],numeric[::max(1,cells//15)],"o",ms=3,color=blue,label="Finite differences")
ax.legend()
plt.show()

The finite-difference solution follows the field selected by the grounded exterior wall. Its agreement with that curve says nothing about whether the wall models open space.

In [ ]:
R=np.geomspace(1.03*probe_ratio,max(30,5*probe_ratio),140)
fig,ax=axes("Outer radius R/a","Relative probe-amplitude error")
ax.loglog(R,1-ratio(probe_ratio,R),color=teal,label="Exact boundary error")
ax.loglog(R,(probe_ratio**3-1)/R**3,"--",color=orange,label="Leading inverse cube")
ax.legend()
plt.show()

Increasing the domain radius at fixed physical probe location shows the inverse-cube asymptote. A mesh study alone would miss this error.

## Exercise · size the exterior shell

For a probe at $2a$, require an error below one percent. What radius is needed?

The exact relative bias magnitude is $7/[(R/a)^3-1]$. Hence $(R/a)^3>701$ or $R/a>8.883$. A value of nine satisfies the condition before numerical discretisation is considered.

## Exercise · change the angular dependence

What Robin coefficient would match a pure exterior quadrupole?

A decaying multipole of order $\ell$ behaves as $r^{-(\ell+1)}$. Its logarithmic slope at $R$ is $-(\ell+1)/R$, so the quadrupole needs $3/R$. A superposition of orders requires angular information or an exterior operator that acts differently on each mode.